In [ ]:
import climate_indices
import climate_library
from climate_library.climate_index import ClimateIndex

In [ ]:
import xclim
import xclim.indices
import xclim.core.units as xu
from xclim.testing import open_dataset
from xclim.indices import standardized_precipitation_index
from xclim.indices.stats import standardized_index_fit_params
from xclim.core.calendar import percentile_doy

In [ ]:
import pandas as pd
import geopandas as gpd

In [ ]:
from netCDF4 import Dataset

In [ ]:
import xarray as xr
import numpy as np
from datetime import datetime
from scipy import stats as st
from tqdm import tqdm

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:
import climate_V2

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
ds = xr.open_dataset("netcdf.nc")
ds = ds.rename({'oldname', 'newname'})
ds.to_netcdf('newnewcdf.nc')

In [ ]:
ds = xr.open_dataset("netcdf.nc")
ds = ds.rename(oldname='newname')
ds.to_netcdf('newnewcdf.nc')

In [ ]:
ds = open_dataset(precipitation)
ds['tp'].attrs['units'] = 'mm/day'
ds['tp'].values*1000
ds.to_netcdf('convert_precipitation.nc')

In [ ]:
import netCDF4 as nc

# Open the NetCDF file in read/write mode
dataset = nc.Dataset('your_file.nc', 'r+')

# Rename the coordinate
dataset.renameDimension('valid_time', 'time')

# Save the changes
dataset.close()

In [ ]:
import netCDF4 as nc

# Open the NetCDF file in read/write mode
dataset = nc.Dataset('your_file.nc', 'r+')

# Rename the coordinate
dataset.renameVariable('valid_time', 'time')

# Save the changes
dataset.close()

In [ ]:
import netCDF4 as nc

# Open the NetCDF file in read/write mode
dataset = nc.Dataset('your_file.nc', 'r+')

# Transpose the data variable to a coordinate
dataset.transpose('valid_time', 'time')

# Save the changes
dataset.close()

In [ ]:
# Standardized Precipitation Index Function
def spi(ds, thresh, dimension):
    #ds - data ; thresh - time interval / scale; dimension - dimension as a string

    #Rolling Mean / Moving Averages
    ds_ma = ds.rolling(time = thresh, center=False).mean(dim=dimension)

    #Natural log of moving averages
    ds_In = np.log(ds_ma)
    ds_In = ds_In.where(np.isinf(ds_In) == False) #= np.nan  #Change infinity to NaN

    #Overall Mean of Moving Averages
    ds_mu = ds_ma.mean(dimension)

    #Summation of Natural log of moving averages
    ds_sum = ds_In.sum(dimension)

    #Computing essentials for gamma distribution
    n = ds_In[thresh-1:, :, :].count(dimension)                  #size of data

    A = np.log(ds_mu) - (ds_sum/n)             #Computing A
    alpha = (1/(4*A))*(1+(1+((4*A)/3))**0.5)   #Computing alpha  (a)
    beta = ds_mu/alpha                         #Computing beta (scale)
    
    #Gamma Distribution (CDF) 
    gamma_func = lambda data, a, scale: st.gamma.cdf(data, a=a, scale=scale)
    gamma = xr.apply_ufunc(gamma_func, ds_ma, alpha, beta)
    
    #Standardized Precipitation Index   (Inverse of CDF)
    norminv = lambda data: st.norm.ppf(data, loc=0, scale=1)
    norm_spi = xr.apply_ufunc(norminv, gamma)  #loc is mean and scale is standard dev.
    
    return ds_ma, ds_In , ds_mu, ds_sum,n, A, alpha, beta, gamma, norm_spi

da_data = xr.open_dataset('C:/Netcdf/cru_ts4.08.1901.2023.pre.dat.nc')
ds_RR = da_data['pre']
# ds_RR_Thailand= ds_RR.sel(lon=slice(96, 106), lat=slice(4, 21),time=slice('2015','2018'))
# ds_RR_Thailand= ds_RR.sel(lon=slice(96, 106), lat=slice(4, 21),time=slice('1901', '1902'))
ds_RR_Thailand= ds_RR.sel(lon=slice(96, 106), lat=slice(4, 21),time='1901')
i=3
ddata = spi(ds_RR_Thailand,i,'time')[9]
ddata.plot(cmap='RdBu', col='time', col_wrap=4, vmin=-2.5, vmax=2.5)

# print(ddata)
plt.show()

In [ ]:
#Standardized Precipitation Index Function
def spi(ds, thresh):
    #ds - data ; thresh - time interval / scale
    
    #Rolling Mean / Moving Averages
    ds_ma = ds.rolling(thresh, center=False).mean()
    
    #Natural log of moving averages
    ds_In = np.log(ds_ma)
    ds_In[ np.isinf(ds_In) == True] = np.nan  #Change infinity to NaN
    
    #Overall Mean of Moving Averages
    ds_mu = np.nanmean(ds_ma)
    
    #Summation of Natural log of moving averages
    ds_sum = np.nansum(ds_In)
        
    #Computing essentials for gamma distribution
    n = len(ds_In[thresh-1:])                  #size of data
    A = np.log(ds_mu) - (ds_sum/n)             #Computing A
    alpha = (1/(4*A))*(1+(1+((4*A)/3))**0.5)   #Computing alpha  (a)
    beta = ds_mu/alpha                         #Computing beta (scale)
    
    #Gamma Distribution (CDF)
    gamma = st.gamma.cdf(ds_ma, a=alpha, scale=beta)  
    
    #Standardized Precipitation Index   (Inverse of CDF)
    norm_spi = st.norm.ppf(gamma, loc=0, scale=1)  #loc is mean and scale is standard dev.
    
    return ds_ma, ds_In, ds_mu, ds_sum, n, A, alpha, beta, gamma, norm_spi

data = pd.read_csv('precipitation.csv', usecols=[1])

data = data.set_index(pd.date_range('1901', '2024', freq='M'))
times = [3, 6, 9, 12, 24]
for i in times:
    x = spi(data['station1-97.46153389044547-18.4142241806728'], i)
    data['spi_'+str(i)] = x[9]

fig, axes = plt.subplots(nrows=5, figsize=(15, 10))
plt.subplots_adjust(hspace=0.15)
for i, ax in enumerate(axes):
    col_scheme=np.where(data['spi_'+str(times[i])]>0, 'b','r')

    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.bar(data.index, data['spi_'+str(times[i])], width=25, align='center', color=col_scheme, label='SPI '+str(times[i]))
    ax.axhline(y=0, color='k')
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.legend(loc='upper right')
    ax.set_yticks(range(-3,4), range(-3,4))
    ax.set_ylabel('SPI', fontsize=12)
    
    if i<len(times)-1:
        ax.set_xticks([],[])

plt.show()

In [ ]:
da_data = xr.open_dataset('C:/Netcdf/cru_ts4.08.1901.2023.pre.dat.nc')
ds_RR = da_data['pre']

ds_RR_Thailand= ds_RR.sel(lon=slice(96, 106), lat=slice(4, 21),time='1901')
i=3

test = climate_V2.Climate(ds_RR_Thailand)
ddata = test.calculate_spi(thresh=i,dimension='time',precip_var='pre')

ddata[9].plot(cmap='RdBu', col='time', col_wrap=4, vmin=-2.5, vmax=2.5)
plt.show()